In [ ]:

import pandas as pd

# 加载训练集数据
train_file_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/train.csv'
train_df = pd.read_csv(train_file_path)

# 查看数据的前几行
print(train_df.head())

# 检查数据的基本信息
print(train_df.info())

# 检查缺失值
print(train_df.isnull().sum())


  surgery  hospital_number  ...  capillary_refill_time     outcome
0     yes           527706  ...             less_3_sec        died
1     yes           528641  ...             less_3_sec       lived
2     yes           535043  ...             more_3_sec  euthanized
3     yes           535043  ...             less_3_sec  euthanized
4     yes           528890  ...             more_3_sec        died

[5 rows x 9 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 986 entries, 0 to 985
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery                986 non-null    object 
 1   hospital_number        986 non-null    int64  
 2   rectal_temp            986 non-null    float64
 3   pulse                  986 non-null    float64
 4   respiratory_rate       986 non-null    float64
 5   peripheral_pulse       938 non-null    object 
 6   mucous_membrane        971 non-null    object 
 7  

In [ ]:


# 处理缺失值
# 对于分类特征，可以使用众数填充
train_df['peripheral_pulse'].fillna(train_df['peripheral_pulse'].mode()[0], inplace=True)
train_df['mucous_membrane'].fillna(train_df['mucous_membrane'].mode()[0], inplace=True)
train_df['capillary_refill_time'].fillna(train_df['capillary_refill_time'].mode()[0], inplace=True)

# 检查缺失值是否已经处理
print(train_df.isnull().sum())



surgery                  0
hospital_number          0
rectal_temp              0
pulse                    0
respiratory_rate         0
peripheral_pulse         0
mucous_membrane          0
capillary_refill_time    0
outcome                  0
dtype: int64
C:\Users\xuyutian\AppData\Local\Temp\ipykernel_4412\345476409.py:6: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  train_df['peripheral_pulse'].fillna(train_df['peripheral_pulse'].mode()[0], inplace=True)
C:\Users\xuyutian\AppData\Local\Temp\ipykernel_4412\345476409.py:8: Futu

In [ ]:


from sklearn.preprocessing import LabelEncoder

# 初始化LabelEncoder
le = LabelEncoder()

# 对分类特征进行编码
train_df['surgery'] = le.fit_transform(train_df['surgery'])
train_df['peripheral_pulse'] = le.fit_transform(train_df['peripheral_pulse'])
train_df['mucous_membrane'] = le.fit_transform(train_df['mucous_membrane'])
train_df['capillary_refill_time'] = le.fit_transform(train_df['capillary_refill_time'])
train_df['outcome'] = le.fit_transform(train_df['outcome'])

# 查看编码后的数据
print(train_df.head())


   surgery  hospital_number  ...  capillary_refill_time  outcome
0        1           527706  ...                      0        0
1        1           528641  ...                      0        2
2        1           535043  ...                      1        1
3        1           535043  ...                      0        1
4        1           528890  ...                      1        0

[5 rows x 9 columns]


In [ ]:



from sklearn.preprocessing import StandardScaler

# 初始化StandardScaler
scaler = StandardScaler()

# 对数值特征进行标准化
train_df[['rectal_temp', 'pulse', 'respiratory_rate']] = scaler.fit_transform(train_df[['rectal_temp', 'pulse', 'respiratory_rate']])

# 查看标准化后的数据
print(train_df.head())


   surgery  hospital_number  ...  capillary_refill_time  outcome
0        1           527706  ...                      0        0
1        1           528641  ...                      0        2
2        1           535043  ...                      1        1
3        1           535043  ...                      0        1
4        1           528890  ...                      1        0

[5 rows x 9 columns]


In [ ]:



from sklearn.model_selection import train_test_split

# 分离特征和标签
X = train_df.drop(columns=['outcome'])
y = train_df['outcome']

# 划分训练集和验证集
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

# 查看训练集和验证集的大小
print(f"训练集大小: {X_train.shape[0]}")
print(f"验证集大小: {X_val.shape[0]}")



训练集大小: 788
验证集大小: 198


In [ ]:



from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import f1_score

# 初始化随机森林分类器
rf_model = RandomForestClassifier(random_state=42)

# 训练模型
rf_model.fit(X_train, y_train)

# 在验证集上进行预测
y_val_pred = rf_model.predict(X_val)

# 计算F1分数
f1 = f1_score(y_val, y_val_pred, average='weighted')

# 输出F1分数
print(f"验证集上的F1分数: {f1:.4f}")




验证集上的F1分数: 0.6090


In [ ]:




from sklearn.model_selection import GridSearchCV

# 定义超参数网格
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 10, 20, 30],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

# 初始化GridSearchCV
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=5, scoring='f1_weighted', n_jobs=-1)

# 进行网格搜索
grid_search.fit(X_train, y_train)

# 输出最佳参数
print(f"最佳参数: {grid_search.best_params_}")

# 使用最佳参数的模型进行预测
y_val_pred_best = grid_search.best_estimator_.predict(X_val)

# 计算最佳模型在验证集上的F1分数
f1_best = f1_score(y_val, y_val_pred_best, average='weighted')

# 输出F1分数
print(f"验证集上的F1分数（最佳模型）: {f1_best:.4f}")





最佳参数: {'max_depth': 10, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 200}
验证集上的F1分数（最佳模型）: 0.6072


In [ ]:



# 加载测试集数据
test_file_path = r'D:\LLM-Driven_AI-Studio\MLAgent\data\benchmark/NL2Workflow/datasets/tabular/horse_health_outcomes/test.csv'
test_df = pd.read_csv(test_file_path)

# 检查测试集数据的基本信息
print(test_df.head())
print(test_df.info())
print(test_df.isnull().sum())


  surgery  hospital_number  ...  capillary_refill_time     outcome
0      no           535381  ...             less_3_sec  euthanized
1     yes           535029  ...             less_3_sec  euthanized
2     yes           529461  ...             more_3_sec        died
3     yes           534157  ...             less_3_sec  euthanized
4     yes           529777  ...             less_3_sec       lived

[5 rows x 9 columns]
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 247 entries, 0 to 246
Data columns (total 9 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   surgery                247 non-null    object 
 1   hospital_number        247 non-null    int64  
 2   rectal_temp            247 non-null    float64
 3   pulse                  247 non-null    float64
 4   respiratory_rate       247 non-null    float64
 5   peripheral_pulse       235 non-null    object 
 6   mucous_membrane        241 non-null    object 
 7  

In [ ]:




# 使用训练集的众数填充测试集的缺失值
test_df['peripheral_pulse'].fillna(train_df['peripheral_pulse'].mode()[0], inplace=True)
test_df['mucous_membrane'].fillna(train_df['mucous_membrane'].mode()[0], inplace=True)
test_df['capillary_refill_time'].fillna(train_df['capillary_refill_time'].mode()[0], inplace=True)

# 检查缺失值是否已经处理
print(test_df.isnull().sum())

# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['surgery'] = le.transform(test_df['surgery'])
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])
test_df['outcome'] = le.transform(test_df['outcome'])

# 使用训练集的StandardScaler 对数值特征进行标准化
test_df[['rectal_temp', 'pulse', 'respiratory_rate']] = scaler.transform(test_df[['rectal_temp', 'pulse', 'respiratory_rate']])

# 查看预处理后的测试集数据
print(test_df.head())


Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

avior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  test_df['capillary_refill_time'].fillna(train_df['capillary_refill_time'].mode()[0], inplace=True)
---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:235, in _encode(values, uniques, check_unknown)
    234 try:
--> 235     return _map_to_integ

In [ ]:




# 手动处理未见过的标签
test_df['surgery'] = test_df['surgery'].map({'yes': 1, 'no': 0}).astype(int)
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])
test_df['outcome'] = le.transform(test_df['outcome'])

# 查看预处理后的测试集数据
print(test_df.head())


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:235, in _encode(values, uniques, check_unknown)
    234 try:
--> 235     return _map_to_integer(values, uniques)
    236 except KeyError as e:

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:174, in _map_to_integer(values, uniques)
    173 table = _nandict({val: i for i, val in enumerate(uniques)})
--> 174 return xp.asarray([table[v] for v in values], device=device(values))

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:174, in <listcomp>(.0)
    173 table = _nandict({val: i for i, val in enumerate(uniques)})
--> 174 return xp.asarray([table[v] for v in values], device=device(values))

File D:\L

In [ ]:




# 手动处理未见过的标签
test_df['surgery'] = test_df['surgery'].map({'yes': 1, 'no': 0}).astype(int)

# 找出测试集中未见过的标签
unseen_peripheral_pulse = set(test_df['peripheral_pulse']) - set(le.classes_)
unseen_mucous_membrane = set(test_df['mucous_membrane']) - set(le.classes_)
unseen_capillary_refill_time = set(test_df['capillary_refill_time']) - set(le.classes_)

# 将未见过的标签映射到一个新的类别（例如 -1）
test_df['peripheral_pulse'] = test_df['peripheral_pulse'].apply(lambda x: x if x in le.classes_ else -1)
test_df['mucous_membrane'] = test_df['mucous_membrane'].apply(lambda x: x if x in le.classes_ else -1)
test_df['capillary_refill_time'] = test_df['capillary_refill_time'].apply(lambda x: x if x in le.classes_ else -1)

# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])
test_df['outcome'] = le.transform(test_df['outcome'])

# 查看预处理后的测试集数据
print(test_df.head())



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

esult_blocks = extend_blocks(applied, result_blocks)
    366 out = type(self).from_blocks(result_blocks, self.axes)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\internals\blocks.py:758, in Block.astype(self, dtype, copy, errors, using_cow, squeeze)
    755         raise ValueError("Can not squeeze with more than one column.")
    756     values = values[0, :]  # type: ignore[call-overload]
--> 758 new_values = astype_array_safe(values, dtype, copy=copy, errors=errors)
    760 new_values = maybe_coerce_values(new_values)
    762 refs = None

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\dtypes\astype.py:237, in astype_array_safe(values, dtype, copy, errors)
    234     dtype = dtype.numpy_dtype
    236 try:
--> 237 

In [ ]:





# 处理非有限值
test_df['outcome'] = test_df['outcome'].fillna(-1).astype(int)

# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])
test_df['outcome'] = le.transform(test_df['outcome'])

# 查看预处理后的测试集数据
print(test_df.head())



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

urn res.__finalize__(self, method="astype")

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\internals\managers.py:430, in BaseBlockManager.astype(self, dtype, copy, errors)
    427 elif using_copy_on_write():
    428     copy = False
--> 430 return self.apply(
    431     "astype",
    432     dtype=dtype,
    433     copy=copy,
    434     errors=errors,
    435     using_cow=using_copy_on_write(),
    436 )

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\internals\managers.py:363, in BaseBlockManager.apply(self, f, align_keys, **kwargs)
    361         applied = b.apply(f, **kwargs)
    362     else:
--> 363         applied = getattr(b, f)(**kwargs)
    364     result_blocks = extend_blocks(applied, result_blocks)
  

In [ ]:






# 手动处理未见过的标签
test_df['surgery'] = test_df['surgery'].map({'yes': 1, 'no': 0}).astype(int)

# 找出测试集中未见过的标签
unseen_peripheral_pulse = set(test_df['peripheral_pulse']) - set(le.classes_)
unseen_mucous_membrane = set(test_df['mucous_membrane']) - set(le.classes_)
unseen_capillary_refill_time = set(test_df['capillary_refill_time']) - set(le.classes_)

# 将未见过的标签映射到一个新的类别（例如 -1）
test_df['peripheral_pulse'] = test_df['peripheral_pulse'].apply(lambda x: x if x in le.classes_ else -1)
test_df['mucous_membrane'] = test_df['mucous_membrane'].apply(lambda x: x if x in le.classes_ else -1)
test_df['capillary_refill_time'] = test_df['capillary_refill_time'].apply(lambda x: x if x in le.classes_ else -1)

# 手动处理 outcome 列中的未见过的标签
test_df['outcome'] = test_df['outcome'].map({'lived': 0, 'died': 1, 'euthanized': -1}).astype(int)

# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])
test_df['outcome'] = test_df['outcome']

# 查看预处理后的测试集数据
print(test_df.head())




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

esult_blocks = extend_blocks(applied, result_blocks)
    366 out = type(self).from_blocks(result_blocks, self.axes)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\internals\blocks.py:758, in Block.astype(self, dtype, copy, errors, using_cow, squeeze)
    755         raise ValueError("Can not squeeze with more than one column.")
    756     values = values[0, :]  # type: ignore[call-overload]
--> 758 new_values = astype_array_safe(values, dtype, copy=copy, errors=errors)
    760 new_values = maybe_coerce_values(new_values)
    762 refs = None

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\dtypes\astype.py:237, in astype_array_safe(values, dtype, copy, errors)
    234     dtype = dtype.numpy_dtype
    236 try:
--> 237 

In [ ]:



# 检查 outcome 列中的唯一值
print(test_df['outcome'].unique())

# 确保 outcome 列中的值都被正确处理
test_df['outcome'] = test_df['outcome'].map({'lived': 0, 'died': 1, 'euthanized': -1}).fillna(-1).astype(int)

# 查看处理后的 outcome 列
print(test_df['outcome'].head())


['euthanized' 'died' 'lived']
0   -1
1   -1
2    1
3   -1
4    0
Name: outcome, dtype: int64


In [ ]:




# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])

# 使用训练集的StandardScaler 对数值特征进行标准化
test_df[['rectal_temp', 'pulse', 'respiratory_rate']] = scaler.transform(test_df[['rectal_temp', 'pulse', 'respiratory_rate']])

# 查看预处理后的测试集数据
print(test_df.head())


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:235, in _encode(values, uniques, check_unknown)
    234 try:
--> 235     return _map_to_integer(values, uniques)
    236 except KeyError as e:

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:174, in _map_to_integer(values, uniques)
    173 table = _nandict({val: i for i, val in enumerate(uniques)})
--> 174 return xp.asarray([table[v] for v in values], device=device(values))

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:174, in <listcomp>(.0)
    173 table = _nandict({val: i for i, val in enumerate(uniques)})
--> 174 return xp.asarray([table[v] for v in values], device=device(values))

File D:\L

In [ ]:




# 手动处理未见过的标签
test_df['peripheral_pulse'] = test_df['peripheral_pulse'].map({'reduced': 0, 'weak': 1, 'normal': -1}).astype(int)

# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['peripheral_pulse'] = le.transform(test_df['peripheral_pulse'])
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])

# 使用训练集的StandardScaler 对数值特征进行标准化
test_df[['rectal_temp', 'pulse', 'respiratory_rate']] = scaler.transform(test_df[['rectal_temp', 'pulse', 'respiratory_rate']])

# 查看预处理后的测试集数据
print(test_df.head())



Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

esult_blocks = extend_blocks(applied, result_blocks)
    366 out = type(self).from_blocks(result_blocks, self.axes)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\internals\blocks.py:758, in Block.astype(self, dtype, copy, errors, using_cow, squeeze)
    755         raise ValueError("Can not squeeze with more than one column.")
    756     values = values[0, :]  # type: ignore[call-overload]
--> 758 new_values = astype_array_safe(values, dtype, copy=copy, errors=errors)
    760 new_values = maybe_coerce_values(new_values)
    762 refs = None

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\dtypes\astype.py:237, in astype_array_safe(values, dtype, copy, errors)
    234     dtype = dtype.numpy_dtype
    236 try:
--> 237 

In [ ]:




# 检查 peripheral_pulse 列中的唯一值
print(test_df['peripheral_pulse'].unique())

# 确保 peripheral_pulse 列中的值都被正确处理
test_df['peripheral_pulse'] = test_df['peripheral_pulse'].map({'reduced': 0, 'weak': 1, 'normal': -1}).fillna(-1).astype(int)

# 查看处理后的 peripheral_pulse 列
print(test_df['peripheral_pulse'].head())



['normal' 'reduced' np.int64(3) 'absent' 'increased']
0   -1
1   -1
2    0
3    0
4   -1
Name: peripheral_pulse, dtype: int64


In [ ]:





# 使用训练集的LabelEncoder 对分类特征进行编码
test_df['mucous_membrane'] = le.transform(test_df['mucous_membrane'])
test_df['capillary_refill_time'] = le.transform(test_df['capillary_refill_time'])

# 使用训练集的StandardScaler 对数值特征进行标准化
test_df[['rectal_temp', 'pulse', 'respiratory_rate']] = scaler.transform(test_df[['rectal_temp', 'pulse', 'respiratory_rate']])

# 查看预处理后的测试集数据
print(test_df.head())


---------------------------------------------------------------------------
KeyError                                  Traceback (most recent call last)
File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:235, in _encode(values, uniques, check_unknown)
    234 try:
--> 235     return _map_to_integer(values, uniques)
    236 except KeyError as e:

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:174, in _map_to_integer(values, uniques)
    173 table = _nandict({val: i for i, val in enumerate(uniques)})
--> 174 return xp.asarray([table[v] for v in values], device=device(values))

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\sklearn\utils\_encode.py:174, in <listcomp>(.0)
    173 table = _nandict({val: i for i, val in enumerate(uniques)})
--> 174 return xp.asarray([table[v] for v in values], device=device(values))

File D:\L